In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
# import xgboost as xgb
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.logei import qLogNoisyExpectedImprovement
import seaborn

In [2]:
o = 27
n = 9
s = o-n
sampler = Sampler_class()
Parameters_lis = [
    RangeParameterConfig(name="s1", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="s2", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="b1", parameter_type="float", bounds=tuple([0,1])),
    ]

In [3]:
client = Client()
gp_model = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/Modelling/ModelMk4.json")
gp_model.get_next_trials(max_trials=1)
def SurrogateModelOfReality(s1, s2, b1):
    y_pred = gp_model.predict([{"s1":s1,"s2":s2,"b1":b1}])[0]["t1"][0]
    return np.float64(y_pred)

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


In [4]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(
            name="s1", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="s2", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="b1", parameter_type="float", bounds=(0, 1)
        ),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": qLogNoisyExpectedImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)

    X = sampler.three.PseudorandomSampler3D_func(n,Parameters_lis).T

    for array in X:
        my_parameters = {"s1": array[0], "s2": array[1], "b1": array[2]}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(**my_parameters)})

    for _ in range(s): # Run 10 rounds of trials
        # We will request three trials at a time in this example
        trials = client.get_next_trials(max_trials=1)

        for trial_index, parameters in trials.items():
            s1 = parameters["s1"]
            s2 = parameters["s2"]
            b1 = parameters["b1"]

            result = SurrogateModelOfReality(s1, s2, b1)

            # Set raw_data as a dictionary with metric names as keys and results as values
            raw_data = {metric_name: result}

            # Complete the trial with the result
            client.complete_trial(trial_index=trial_index, raw_data=raw_data)
    # print(client.summarize())
    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(client.summarize().t1))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

Trial 0 =========================================
15.387155991269474

Trial 1 =========================================
15.388557461040236

Trial 2 =========================================
15.399759863568285

Trial 3 =========================================
16.27403733412312

Trial 4 =========================================
14.320556960907062

Trial 5 =========================================
15.392738437475646

Trial 6 =========================================
16.254649508062695

Trial 7 =========================================
13.404709109348882

Trial 8 =========================================
14.157186467591847



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/botorch/optim/optimize.py:331: BadInitialCandidatesWarning: Unable to find non-zero acquisition function values - initial conditions are being selected randomly.
  generated_initial_conditions = opt_inputs.get_ic_generator()(


Trial 9 =========================================
16.16170745565479

Trial 10 =========================================
13.412503022664016

Trial 11 =========================================
15.381282912176987

Trial 12 =========================================
18.16552676016302



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/gpytorch/distributions/multivariate_normal.py:376: NumericalWarning: Negative variance values detected. This is likely due to numerical instabilities. Rounding negative variances up to 1e-10.
  warnings.warn(


Trial 13 =========================================
15.71855184615373

Trial 14 =========================================
14.852697166883795

Trial 15 =========================================
16.213322875007776

Trial 16 =========================================
15.397201727278647

Trial 17 =========================================
15.361929601789814

Trial 18 =========================================
15.39448470948182

Trial 19 =========================================
15.328853151860454

Trial 20 =========================================
18.802420328936705

Trial 21 =========================================
13.21054627003746

Trial 22 =========================================
18.746615797633897

Trial 23 =========================================
15.353859213008299

Trial 24 =========================================
17.591658862882205



...

Trial 86 =========================================
16.208207888160114

Trial 87 =========================================
14.43738020434553

Trial 88 =========================================
16.190778539957954

Trial 89 =========================================
15.929222010399386

Trial 90 =========================================
14.440766724092848

Trial 91 =========================================
18.762085012693518

Trial 92 =========================================
14.292966734869513

Trial 93 =========================================
16.24196243601155



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 94 =========================================
17.41782560560709

Trial 95 =========================================
18.77877267874197

Trial 96 =========================================
18.80951467518469

Trial 97 =========================================
14.466543831528037



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 98 =========================================
15.929222010399386

Trial 99 =========================================
16.01208340326966

[15.38715599 15.38855746 15.39975986 16.27403733 14.32055696 15.39273844
 16.25464951 13.40470911 14.15718647 16.16170746 13.41250302 15.38128291
 18.16552676 15.71855185 14.85269717 16.21332288 15.39720173 15.3619296
 15.39448471 15.32885315 18.80242033 13.21054627 18.7466158  15.35385921
 17.59165886 16.24223865 15.35388033 18.72718619 15.35063145 16.4358898
 14.63371725 16.03871309 17.24856095 16.24344635 15.39059637 15.39247734
 15.31552117 13.87511656 14.25202071 16.13797405 16.24282883 15.28219994
 15.39537679 15.36288363 15.39136511 16.21733076 14.39714874 15.36503557
 14.23680635 15.39022449 14.44355112 14.16375111 14.28746208 15.38913015
 14.14991682 18.77997701 15.34493457 15.3740952  16.19388598 17.58256542
 15.38940251 18.81633423 14.30621875 13.98700201 15.35164599 14.62041575
 17.55147538 16.00854997 18.74298809 18.82083296 14.928023

In [5]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 18.820832956642946
Avg = 15.812394219540353
Std = 1.3964115325730009


In [6]:
print(y_max_arr.tolist())

[15.387155991269474, 15.388557461040236, 15.399759863568285, 16.27403733412312, 14.320556960907062, 15.392738437475646, 16.254649508062695, 13.404709109348882, 14.157186467591847, 16.16170745565479, 13.412503022664016, 15.381282912176987, 18.16552676016302, 15.71855184615373, 14.852697166883795, 16.213322875007776, 15.397201727278647, 15.361929601789814, 15.39448470948182, 15.328853151860454, 18.802420328936705, 13.21054627003746, 18.746615797633897, 15.353859213008299, 17.591658862882205, 16.24223864737382, 15.353880331052224, 18.7271861860281, 15.350631450585658, 16.43588980252131, 14.633717254132408, 16.038713089554005, 17.248560951227034, 16.243446354272102, 15.390596370545408, 15.39247734135419, 15.315521167792568, 13.875116555837844, 14.252020712271015, 16.13797405442282, 16.2428288283731, 15.282199935837209, 15.395376794095435, 15.362883626402457, 15.39136510990299, 16.217330759930906, 14.397148744995073, 15.365035571806063, 14.236806350158387, 15.390224490595893, 14.44355112145

In [7]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/SequentialTestingOfSamplingTechnique/DataGenerated/BOpt-EI-9,27,1-9.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)